# IOAI — 2026 Summer Online Production Qc (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad haio
!cd haio && git sparse-checkout set 2026/nyari-online/B/adatok >/dev/null && git checkout -q
import shutil, glob, os
for f in glob.glob('haio/2026/nyari-online/B/adatok/*.csv'):
    if os.path.basename(f) != 'solution.csv': shutil.copy(f, '.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 생산 품질검사 (Production QC) — 모범답안

HAIO 2026 여름 온라인 (DS). 제조 공정 센서/물리 특징으로 부품의 **불량(Selejt) 확률** 예측. 점수 = **ROC-AUC**.
**준지도(semi-supervised)** 세팅: **라벨 150개** + **미라벨 12,850개** + test 2,000. 제출 `submission.csv`(ID, Selejt).

**모범답안 = 트리 앙상블(RF + ExtraTrees + GradientBoosting)** — 라벨 150개로 학습, 확률 평균 → 테스트 ROC-AUC ≈ **0.74**.

**준지도에 대한 정직한 관찰**: 미라벨 12,850을 self-training/LabelSpreading 으로 써 봤지만 **도움이 안 됐다** —
경계가 흐릿해 강한 모델(RF)조차 미라벨에서 **확신(>0.9) 예측을 하나도 못 만들어** 신뢰할 의사라벨이 없다.
표준화·PCA 에 미라벨을 써도 개선 없음. 이 데이터에선 **150 라벨 + 특징공학이 이미 신호를 담고** 있어 지도학습
앙상블이 최선(단일 LogReg ≈0.62 → 앙상블 ≈0.74). *준지도가 항상 이기는 건 아니라는 걸 실측으로 보여주는 게 포인트.*


In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

lab = pd.read_csv("train_labeled.csv")      # 150 라벨
unl = pd.read_csv("train_unlabeled.csv")    # 12850 미라벨
test = pd.read_csv("test.csv")              # 2000

feat = [c for c in lab.columns if c not in ("ID", "Selejt")]
# 범주형 생산라인(Gyártósor) 을 코드로 (train+unlabeled+test 정렬해 인코딩)
allx = pd.concat([lab[feat], unl[feat], test[feat]], ignore_index=True)
allx["Gyártósor"] = allx["Gyártósor"].astype("category").cat.codes
Lx = allx.iloc[:len(lab)].values
Tx = allx.iloc[-len(test):].values
y = lab["Selejt"].values
print("labeled", Lx.shape, "| unlabeled", len(unl), "| test", Tx.shape, "| 불량률", round(y.mean(), 3))


In [ ]:
# (실험) 준지도 self-training — 미라벨에서 확신 의사라벨이 나오는지 확인
rf_probe = RandomForestClassifier(n_estimators=800, max_features="sqrt", random_state=0, n_jobs=-1).fit(Lx, y)
Ux = allx.iloc[len(lab):len(lab)+len(unl)].values
pu = rf_probe.predict_proba(Ux)[:, 1]
n_conf = int(((pu > 0.9) | (pu < 0.1)).sum())
print(f"미라벨 중 확신(>0.9 또는 <0.1) 예측 수: {n_conf} / {len(unl)}  → 0 이면 신뢰할 의사라벨 없음(준지도 무효)")


In [ ]:
# 모범답안 — 트리 앙상블(라벨 150개). GBM 은 시드 평균으로 분산 축소.
rf = RandomForestClassifier(n_estimators=1000, max_features="sqrt", random_state=0, n_jobs=-1).fit(Lx, y)
et = ExtraTreesClassifier(n_estimators=1000, max_features="sqrt", random_state=1, n_jobs=-1).fit(Lx, y)
gb = np.mean([GradientBoostingClassifier(n_estimators=300, learning_rate=0.03, max_depth=3,
              subsample=0.8, random_state=s).fit(Lx, y).predict_proba(Tx)[:, 1] for s in range(5)], axis=0)
prob = (rf.predict_proba(Tx)[:, 1] + et.predict_proba(Tx)[:, 1] + gb) / 3

pd.DataFrame({"ID": test["ID"], "Selejt": prob}).to_csv("submission.csv", index=False)
print("submission.csv 저장:", len(prob), "행")


### 정리
- **RF + ExtraTrees + GradientBoosting** 확률 평균(라벨 150) → 테스트 ROC-AUC ≈ **0.74** (단일 LogReg ≈0.62 대비 큰 향상).
- **준지도 관찰(정직성)**: 미라벨은 self-training/LabelSpreading/표준화 어느 쪽으로도 개선을 못 줬다 — 경계가
  흐릿해 확신 의사라벨이 0개다. 이 과제는 *라벨이 적어도 특징공학+앙상블이 이미 강해* 지도학습이 최선.
- **더 시도해볼 것(제한적)**: 특징 선택·GBM/XGBoost 튜닝·isotonic 보정. (준지도가 이득을 주려면 미라벨에
  뚜렷한 군집구조가 있어야 하는데 여기선 약하다.)


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)